# Feature Store Iceberg Table Properties

Create a Feature Group with Iceberg table format and manage Iceberg table properties using `FeatureGroupManager`. This notebook demonstrates creating, retrieving, and updating Iceberg properties on Feature Store offline stores.

## Prerequisites

- An AWS account with SageMaker Feature Store access
- An S3 bucket for offline store data
- IAM role with AmazonSageMakerFeatureStoreAccess and AmazonSageMakerFullAccess policies
- `pyiceberg[glue]>=0.8.0` installed
- `AWS_DEFAULT_REGION` set if not using passing a region in function calls

## What This Notebook Does

1. Prepare Sample Data and Feature Definitions
2. Creates a Feature Group with Iceberg table format and initial Iceberg properties
    - Shows IcebergProperties validation
3. Retrieves the Feature Group with its Iceberg properties
4. Updates Iceberg properties on an existing Feature Group
5. Verifies the updated properties
6. Cleans up resources
7. Allowed Iceberg Properties Reference

In [ ]:
# Imports
import time
from datetime import datetime, timezone, timedelta

import boto3
import pandas as pd
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.utils import unique_name_from_base
from sagemaker.mlops.feature_store import (
    FeatureGroupManager,
    OfflineStoreConfig,
    S3StorageConfig,
)
from sagemaker.mlops.feature_store.feature_group_manager import IcebergProperties
from sagemaker.mlops.feature_store.feature_utils import load_feature_definitions_from_dataframe

In [ ]:
# Create Sagemaker Session
session = Session()
role = get_execution_role()
region = boto3.Session().region_name
bucket = session.default_bucket()

print(f"Role: {role}")
print(f"Region: {region}")
print(f"Bucket: {bucket}")

## 1. Prepare Sample Data and Feature Definitions

In [ ]:
feature_group_name = unique_name_from_base("iceberg-properties-example")

base_time = datetime.now(timezone.utc)
df = pd.DataFrame({
    "record_id": [f"id-{i}" for i in range(5)],
    "feature_1": [i * 1.5 for i in range(5)],
    "feature_2": [i * 2 for i in range(5)],
    "event_time": [
        (base_time + timedelta(seconds=i)).strftime("%Y-%m-%dT%H:%M:%SZ")
        for i in range(5)
    ],
})

feature_definitions = load_feature_definitions_from_dataframe(df)
print(f"Feature Group: {feature_group_name}")
print(f"Feature Definitions: {[fd.feature_name for fd in feature_definitions]}")

## 2. Create Feature Group with Iceberg Properties

Create a Feature Group with `table_format='Iceberg'` in the `OfflineStoreConfig` and pass initial Iceberg properties via `IcebergProperties`. The properties are applied to the underlying Glue Iceberg table after the Feature Group reaches `Created` status. 

Objects of type `IcebergProperties` will validate on creation but not mutation so you can optionally validate your iceberg properties before passing them to the `create`, or `update` methods. If you do not validate and your keys are not properly formatted or allowed, then you may recieve an error in your `create` or `update` call.

There is a possibility that the Feature Group creation is successful, but the Iceberg Properties update fails. In this case, please run the udpate function (`fg.update()`) with the iceberg properties to avoid recreating your Feature Group. 

In [ ]:
# Example: invalidate iceberg properties by mutation/addition
invalid_iceberg_props = IcebergProperties(properties={
    "write.metadata.delete-after-commit.enabled": "true",
    "write.metadata.previous-versions-max": "5",
    "history.expire.max-snapshot-age-ms": "86400000",
})
# Add non-allowed property
invalid_iceberg_props.properties.update({"write.delete.isolation-level": "Snapshot"})
try:
    invalid_iceberg_props.validate_property_keys()
except Exception as e:
    print(f"Invalid iceberg props are flagged with the following error: \n{e}")

In [ ]:
#valid iceberg properties
iceberg_props = IcebergProperties(properties={
    "write.metadata.delete-after-commit.enabled": "true",
    "write.metadata.previous-versions-max": "5",
    "history.expire.max-snapshot-age-ms": "86400000",
})

In [ ]:
#Optional: validate properties again
iceberg_props.validate_property_keys()

fg = FeatureGroupManager.create(
    feature_group_name=feature_group_name,
    record_identifier_feature_name="record_id",
    event_time_feature_name="event_time",
    feature_definitions=feature_definitions,
    role_arn=role,
    offline_store_config=OfflineStoreConfig(
        s3_storage_config=S3StorageConfig(
            s3_uri=f"s3://{bucket}/feature-store"
        ),
        table_format="Iceberg",
    ),
    iceberg_properties=iceberg_props,
)

print(f"Feature Group '{feature_group_name}' created with Iceberg properties.")

## 3. Retrieve Feature Group with Iceberg Properties

Use `include_iceberg_properties=True` in `FeatureGroupManager.get()` to fetch the Iceberg table properties from the Glue catalog alongside the Feature Group metadata.

In [ ]:
retrieved_fg = FeatureGroupManager.get(
    feature_group_name=feature_group_name,
    include_iceberg_properties=True,
)

print("Iceberg Properties:")
for key, value in retrieved_fg.iceberg_properties.properties.items():
    print(f"  {key}: {value}")

## 4. Update Iceberg Properties

Update Iceberg properties on an existing Feature Group using `fg.update()`. You can modify existing properties or add new ones. When only `iceberg_properties` is passed, the SageMaker `UpdateFeatureGroup` API call is skipped — only the Glue table properties are updated.

In [ ]:
fg.update(
    iceberg_properties=IcebergProperties(properties={
        "write.metadata.previous-versions-max": "10",
        "write.target-file-size-bytes": "536870912",
    })
)

print("Iceberg properties updated.")

## 5. Verify Updated Properties

In [ ]:
updated_fg = FeatureGroupManager.get(
    feature_group_name=feature_group_name,
    include_iceberg_properties=True,
)

print("Updated Iceberg Properties:")
for key, value in updated_fg.iceberg_properties.properties.items():
    print(f"  {key}: {value}")

## 6. Cleanup

In [ ]:
try:
    fg.delete()
    print(f"Feature Group '{feature_group_name}' deleted.")
except Exception as e:
    print(f"Cleanup error: {e}")

## 7. Allowed Iceberg Properties Reference

The following Iceberg properties are allowed and can be configured on Feature Store offline stores:

| Property | Default Value | Documentation |                                                                                                                                                                                                    
  |----------|---------------|---------------|                                                                                                                                                                                                    
  | write.metadata.delete-after-commit.enabled | FALSE | Controls whether to delete the oldest *tracked* version metadata files after each table commit. |                                                                                        
  | write.metadata.previous-versions-max | 100 | The max number of previous version metadata files to track |                                                                                                                                     
  | history.expire.max-snapshot-age-ms | 432000000 (5 days) | Default max age of snapshots to keep on the table and all of its branches while expiring snapshots |                                                                                
  | history.expire.min-snapshots-to-keep | 1 | Default min number of snapshots to keep on the table and all of its branches while expiring snapshots |                                                                                            
  | write.parquet.row-group-size-bytes | 134217728 (128 MB) | Parquet row group size |                                                                                                                                                            
  | read.split.target-size | 134217728 (128 MB) | Target size when combining data input splits |                                                                                                                                                  
  | read.split.metadata-target-size | 33554432 (32 MB) | Target size when combining metadata input splits |                                                                                                                                       
  | write.delete.target-file-size-bytes | 67108864 (64 MB) | Controls the size of delete files generated to target about this many bytes |                                                                                                        
  | write.delete.mode | copy-on-write | Mode used for delete commands: copy-on-write or merge-on-read (v2 and above) |                                                                                                                            
  | write.update.mode | copy-on-write | Mode used for update commands: copy-on-write or merge-on-read (v2 and above) |                                                                                                                            
  | write.delete.granularity | partition | Controls the granularity of generated delete files: partition or file |                                                                                                                                
  | history.expire.max-ref-age-ms | Long.MAX_VALUE (forever) | For snapshot references except the main branch, default max age of snapshot references to keep while expiring snapshots. The main branch never expires. |                          
  | read.split.open-file-cost | 4194304 (4 MB) | The estimated cost to open a file, used as a minimum weight when combining splits. |                                                                                                             
  | write.target-file-size-bytes | 536870912 (512 MB) | Controls the size of files generated to target about this many bytes |  

  Documentation from https://iceberg.apache.org/docs/latest/configuration/#write-properties